# AI–Quantum Portfolio — Upload One Complete CSV and Run All

Notebook này tải **toàn bộ source code hệ thống từ GitHub**, sau đó chờ người dùng upload một CSV duy nhất. CSV được tách thành price panel, VNAllShare TRI, security master và corporate-action ledger; hệ thống tự dựng universe, chạy quality/leakage audit rồi thực hiện pipeline 33-fold end-to-end.

File chuẩn: `ai_quantum_complete_dataset.csv` (67,6 MB) hoặc `ai_quantum_complete_dataset.zip` (12,7 MB). Notebook không dùng dữ liệu có sẵn trong GitHub làm đầu vào mô hình; chính file upload là nguồn runtime.

In [ ]:
# CẤU HÌNH
from pathlib import Path

REPO_URL = 'https://github.com/23022006muki/AI-Quantum---Finance-Portfolio-Optimization.git'
SOURCE_COMMIT = 'd2843c8a2015783374c222efb50b16aab23be8d2'
EXPECTED_CSV_SHA256 = 'aea9644cfafc359ed04546deca62fea83509826864463669b219f370f1433eba'
RUN_TESTS = True
RUN_FULL_PIPELINE = True
REPO_DIR = Path('/content/AI-Quantum-Finance-Portfolio-Optimization')
PROJECT_DIR = REPO_DIR / 'quantum_portfolio_data'
WORKSPACE = PROJECT_DIR / 'outputs' / 'Data 17_8'
print({'commit': SOURCE_COMMIT, 'run_tests': RUN_TESTS, 'run_full_pipeline': RUN_FULL_PIPELINE})

## 1. Clone riêng source code từ commit đã khóa
Sparse checkout loại các output và dataset GitHub khỏi working tree, bảo đảm dữ liệu mô hình chỉ đến từ file người dùng upload.

In [ ]:
import os, shutil, subprocess, sys

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git','clone','--filter=blob:none','--no-checkout',REPO_URL,str(REPO_DIR)],check=True)
subprocess.run(['git','sparse-checkout','init','--no-cone'],cwd=REPO_DIR,check=True)
patterns = [
    '/quantum_portfolio_data/src/', '/quantum_portfolio_data/scripts/',
    '/quantum_portfolio_data/configs/', '/quantum_portfolio_data/tests/',
    '/quantum_portfolio_data/pyproject.toml', '/quantum_portfolio_data/requirements.lock',
    '/quantum_portfolio_data/app.py', '/quantum_portfolio_data/.gitignore'
]
subprocess.run(['git','sparse-checkout','set',*patterns],cwd=REPO_DIR,check=True)
subprocess.run(['git','checkout',SOURCE_COMMIT],cwd=REPO_DIR,check=True)
actual=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO_DIR,text=True).strip()
assert actual == SOURCE_COMMIT
assert not (PROJECT_DIR/'colab_data').exists(), 'GitHub dataset must not be checked out in upload mode.'
print('Source-only checkout verified:',actual)

## 2. Tạo môi trường Python cô lập và cài thư viện
Colab có hàng trăm package hệ thống với ràng buộc riêng và image Python 3.12 có thể không kèm `ensurepip`. Notebook chỉ cài công cụ PyPA `virtualenv` vào kernel, không thay đổi các thư viện khoa học dựng sẵn và không chạy `pip check` trên môi trường Google; toàn bộ pipeline được cài trong `/content/ai_quantum_venv`, sau đó kiểm tra dependency bên trong môi trường cô lập này.

In [ ]:
VENV_DIR=Path('/content/ai_quantum_venv')
if VENV_DIR.exists(): shutil.rmtree(VENV_DIR)
# Colab Python 3.12 omits ensurepip/python3-venv; virtualenv bundles its own seed wheels.
subprocess.run([sys.executable,'-m','pip','install','--quiet','virtualenv==20.35.4'],check=True)
subprocess.run([sys.executable,'-m','virtualenv',str(VENV_DIR)],check=True)
ENV_PYTHON=VENV_DIR/'bin'/'python'
assert ENV_PYTHON.exists(),ENV_PYTHON
PIPELINE_ENV=os.environ.copy()
PIPELINE_ENV['MPLBACKEND']='Agg'  # Do not inherit Colab's kernel-only matplotlib_inline backend.
subprocess.run([str(ENV_PYTHON),'-m','pip','install','--upgrade','pip','setuptools','wheel'],check=True)
subprocess.run([str(ENV_PYTHON),'-m','pip','install','-e',f'{PROJECT_DIR}[dev]'],check=True)
subprocess.run([str(ENV_PYTHON),'-m','pip','check'],check=True)
health='import numpy,pandas,pyarrow,sklearn,scipy,xgboost,matplotlib,yaml; print("ENV_OK",numpy.__version__,pandas.__version__,pyarrow.__version__,sklearn.__version__,scipy.__version__,xgboost.__version__)'
subprocess.run([str(ENV_PYTHON),'-c',health],env=PIPELINE_ENV,check=True)
print('Isolated pipeline Python:',ENV_PYTHON)

## 3. Upload CSV hoặc ZIP từ máy
Chọn đúng một file. Nếu chọn ZIP, notebook yêu cầu bên trong có đúng một CSV. Đặt `EXPECTED_CSV_SHA256=''` nếu sau này bạn dùng bộ CSV khác đã tự xây dựng.

In [ ]:
from google.colab import files
uploaded=files.upload()
if len(uploaded)!=1:
    raise ValueError('Hãy upload đúng một file CSV hoặc ZIP.')
UPLOAD_PATH=Path('/content')/next(iter(uploaded))
print('Uploaded:',UPLOAD_PATH,f'({UPLOAD_PATH.stat().st_size/1e6:.1f} MB)')

## 4. Giải nén và kiểm tra SHA-256

In [ ]:
import hashlib, zipfile

if UPLOAD_PATH.suffix.lower()=='.zip':
    EXTRACT=Path('/content/uploaded_csv')
    if EXTRACT.exists(): shutil.rmtree(EXTRACT)
    EXTRACT.mkdir()
    with zipfile.ZipFile(UPLOAD_PATH) as zf:
        if zf.testzip(): raise ValueError('ZIP is corrupt.')
        root=EXTRACT.resolve()
        for item in zf.infolist():
            target=(EXTRACT/item.filename).resolve()
            if target!=root and root not in target.parents: raise ValueError('Unsafe ZIP path.')
        zf.extractall(EXTRACT)
    candidates=list(EXTRACT.rglob('*.csv'))
    if len(candidates)!=1: raise ValueError(f'ZIP must contain exactly one CSV; found {len(candidates)}')
    CSV_PATH=candidates[0]
elif UPLOAD_PATH.suffix.lower()=='.csv':
    CSV_PATH=UPLOAD_PATH
else:
    raise ValueError('Only CSV or ZIP is accepted.')

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''): h.update(chunk)
    return h.hexdigest()
CSV_SHA256=sha256(CSV_PATH)
if EXPECTED_CSV_SHA256:
    assert CSV_SHA256.lower()==EXPECTED_CSV_SHA256.lower(),f'Wrong CSV hash: {CSV_SHA256}'
print('CSV:',CSV_PATH)
print('CSV SHA-256:',CSV_SHA256)

## 5. Kiểm tra schema và dựng lại workspace
Importer chỉ chấp nhận đủ năm `record_type`: METADATA, PRICE, BENCHMARK, SECURITY và CORPORATE_ACTION. Nó dựng Parquet, universe monthly, adjustment contract, data quality và leakage audit.

In [ ]:
import json
command=[str(ENV_PYTHON),'scripts/import_colab_complete_csv.py',str(CSV_PATH)]
subprocess.run(command,cwd=PROJECT_DIR,env=PIPELINE_ENV,check=True)
IMPORT_REPORT=json.loads((WORKSPACE/'outputs/reports/COLAB_CSV_IMPORT_REPORT.json').read_text(encoding='utf-8'))
assert IMPORT_REPORT['input_csv_sha256']==CSV_SHA256
assert IMPORT_REPORT['quality_status']=='pass'
assert IMPORT_REPORT['leakage_status'] in {'pass','pass_with_limitations'}
assert IMPORT_REPORT['exploratory_run_permitted']
print(json.dumps(IMPORT_REPORT,indent=2,ensure_ascii=False))

## 6. Hiển thị dữ liệu thực sự được dùng

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown,display,Image

normalized=WORKSPACE/'outputs/normalized'
prices=pd.read_parquet(normalized/'prices.parquet')
benchmark=pd.read_parquet(normalized/'benchmark.parquet')
master=pd.read_parquet(normalized/'security_master_full.parquet')
actions=pd.read_parquet(normalized/'corporate_actions.parquet')
data_table=pd.DataFrame({
 'Chỉ tiêu':['CSV SHA-256','Quan sát giá','Cổ phiếu runtime','Security master','Ngày bắt đầu','Ngày kết thúc','Phiên VNAllShare TRI','Corporate actions','Data quality','Leakage audit'],
 'Kết quả':[CSV_SHA256,len(prices),prices.ticker.nunique(),master.ticker.nunique(),str(prices.date.min().date()),str(prices.date.max().date()),len(benchmark),len(actions),IMPORT_REPORT['quality_status'],IMPORT_REPORT['leakage_status']]
})
display(data_table)
display(prices.head())
display(Markdown('CSV này hoàn chỉnh cho runtime exploratory. Raw PDF và financial-statement PIT không nằm trong CSV nên không được diễn giải là confirmatory full-HOSE.'))

## 7. Chạy test suite và toàn bộ pipeline 33 folds

In [ ]:
if RUN_TESTS:
    subprocess.run([str(ENV_PYTHON),'-m','pytest','-q'],cwd=PROJECT_DIR,env=PIPELINE_ENV,check=True)
experiments=WORKSPACE/'outputs/experiments'
experiments.mkdir(parents=True,exist_ok=True)
before={p.resolve() for p in experiments.iterdir() if p.is_dir()}
if RUN_FULL_PIPELINE:
    subprocess.run([str(ENV_PYTHON),'-m','src.cli','run-data-17-8','--config','configs/data_17_8.yaml'],cwd=PROJECT_DIR,env=PIPELINE_ENV,check=True)
    after={p.resolve() for p in experiments.iterdir() if p.is_dir()}
    created=sorted(after-before,key=lambda p:p.stat().st_mtime)
    if not created: raise RuntimeError('No new experiment was created.')
    ACTIVE=created[-1]
else:
    existing=list(experiments.iterdir())
    if not existing: raise RuntimeError('No experiment in uploaded CSV; set RUN_FULL_PIPELINE=True.')
    ACTIVE=max(existing,key=lambda p:p.stat().st_mtime)
print('ACTIVE EXPERIMENT:',ACTIVE)

## 8. Kết quả tín hiệu, solver và danh mục

In [ ]:
manifest=json.loads((ACTIVE/'manifest.json').read_text(encoding='utf-8'))
rankings=pd.read_csv(ACTIVE/'rankings.csv')
tests=pd.read_csv(ACTIVE/'statistical_tests.csv')
comparisons=pd.read_csv(ACTIVE/'comparisons.csv')
metrics=pd.read_csv(ACTIVE/'strategy_metrics_summary.csv')
fold_rank=rankings.groupby('fold')[['xgboost_rank_ic','ewma_rank_ic']].first()
display(Markdown(f"**Experiment `{manifest['experiment_id']}`:** {manifest['folds_completed']}/{manifest['folds_requested']} folds; OOS {manifest['actual_oos_start']}–{manifest['actual_oos_end']}."))
display(pd.DataFrame({'Model':['XGBoost','EWMA'],'Mean Rank IC':[fold_rank.xgboost_rank_ic.mean(),fold_rank.ewma_rank_ic.mean()],'Median Rank IC':[fold_rank.xgboost_rank_ic.median(),fold_rank.ewma_rank_ic.median()]}).style.format({'Mean Rank IC':'{:.4f}','Median Rank IC':'{:.4f}'}))
labels={'exact':'Exact','simulated_annealing':'Simulated Annealing','penalty_stochastic_baseline':'Penalty stochastic baseline','penalty_qaoa_ideal_statevector':'Penalty-QAOA','xy_qaoa_dicke_ideal_statevector':'XY-QAOA + Dicke'}
solver=comparisons.copy(); solver['Solver']=solver.method.map(labels).fillna(solver.method)
display(solver[['Solver','runs','feasibility_rate','optimality_gap_mean','runtime_seconds']].style.format({'feasibility_rate':'{:.2%}','optimality_gap_mean':'{:.2%}','runtime_seconds':'{:.4f}'}))
preferred=['full_pipeline_xy_qaoa','benchmark_vnallsharetri','liquidity_topk_exact','minimum_variance','equal_weight_universe','ewma_topk_exact','adaptive_exact','xgboost_topk_exact','xgboost_penalty_qaoa']
performance=metrics[metrics.strategy.isin(preferred)].copy(); performance['order']=performance.strategy.map({v:i for i,v in enumerate(preferred)})
performance=performance.sort_values('order')[['strategy','cumulative_return','annualized_return','annualized_volatility','sharpe','max_drawdown','turnover','total_cost']]
display(performance.style.format({'cumulative_return':'{:.2%}','annualized_return':'{:.2%}','annualized_volatility':'{:.2%}','sharpe':'{:.4f}','max_drawdown':'{:.2%}','turnover':'{:.2f}','total_cost':'{:.2%}'}))
for name in ['equity_curve.png','drawdown.png','risk_return.png']:
    path=ACTIVE/'figures'/name
    if path.exists(): display(Image(filename=str(path)))
latest=pd.read_csv(ACTIVE/'latest_selected_portfolio.csv')
summary=json.loads((ACTIVE/'latest_portfolio_summary.json').read_text(encoding='utf-8'))
basket=latest[['ticker','company_name','target_weight','adv_participation']].copy()
basket=pd.concat([basket,pd.DataFrame([{'ticker':'CASH','company_name':'Tiền mặt','target_weight':summary.get('executed_cash_weight',1-basket.target_weight.sum()),'adv_participation':np.nan}])],ignore_index=True)
display(Markdown(f"**Rổ cuối tại {latest.decision_time.iloc[0]}**")); display(basket.style.format({'target_weight':'{:.2%}','adv_participation':'{:.4%}'},na_rep='—'))

## 9. Kết luận kiểm định H1–H6

In [ ]:
def t(name): return tests.loc[tests.test.eq(name)].iloc[0]
h1=t('xgboost_rank_ic_vs_ewma_rank_ic'); h2r=t('adaptive_universe_forward_return_vs_fixed_topm'); h2d=t('adaptive_universe_diversification_vs_fixed_topm'); h3=t('xy_feasibility_vs_penalty_qaoa'); h4=t('xy_optimality_gap_vs_penalty_qaoa'); h5=tests[tests.hypothesis.eq('H5')]
text=f'''### H1 — {'Được hỗ trợ' if h1.p_value_holm<.05 else 'Không được hỗ trợ'}
Δ Rank IC={h1.mean_difference:.4f}, CI [{h1.ci_low:.4f}, {h1.ci_high:.4f}], p-Holm={h1.p_value_holm:.3f}.

### H2 — {'Được hỗ trợ đầy đủ' if h2r.p_value_holm<.05 and h2d.p_value_holm<.05 else 'Được hỗ trợ một phần' if h2r.p_value_holm<.05 or h2d.p_value_holm<.05 else 'Không được hỗ trợ'}
Forward-return p-Holm={h2r.p_value_holm:.3f}; diversification p-Holm={h2d.p_value_holm:.3f}.

### H3 — {'Được hỗ trợ trong ideal simulator' if h3.p_value_holm<.05 else 'Không được hỗ trợ'}
Δ feasibility={h3.mean_difference:.4f}, p-Holm={h3.p_value_holm:.3f}.

### H4 — {'Được hỗ trợ có điều kiện so với Penalty-QAOA' if h4.p_value_holm<.05 else 'Không được hỗ trợ'}
Gap improvement={h4.mean_difference:.4f}, p-Holm={h4.p_value_holm:.3f}; không hàm ý vượt Exact/SA.

### H5 — {'Được hỗ trợ' if (h5.p_value_holm<.05).any() else 'Không được hỗ trợ'}
Có {(h5.p_value_holm<.05).sum()}/{len(h5)} so sánh tài chính đạt ý nghĩa sau Holm.

### H6 — Sensitivity completed
Chỉ suy diễn trong lưới depth, shots, seed, cardinality, noise và chi phí đã khai báo; không phải bằng chứng quantum advantage.'''
display(Markdown(text))

## 10. Tải toàn bộ output về máy

In [ ]:
result_base=Path('/content')/f"ai_quantum_csv_results_{manifest['experiment_id']}"
result_zip=shutil.make_archive(str(result_base),'zip',root_dir=ACTIVE)
print('Created:',result_zip,f'({Path(result_zip).stat().st_size/1e6:.1f} MB)')
display(Markdown(f"Chạy `files.download('{result_zip}')` để tải toàn bộ artifact."))